In [1]:
from numbers import Number

import accelforge as af

In [ ]:
### GPT 175B (target model)

In [ ]:
# Helper functions for running AccelForge evaluations
M = K = N = 8

def make_workload_yaml(density_a=0.25, density_b=0.5, m=8, k=8, n=8):
    """Generate workload YAML with given densities."""
    return {
        'workload': {
            'iteration_space_shape': {
                'm': f'0 <= m < {m}',
                'n': f'0 <= n < {n}',
                'k': f'0 <= k < {k}',
            },
            'bits_per_value': {'All': 8},
            'einsums': [{
                'name': 'SpMSpM',
                'tensor_accesses': [
                    {'name': 'A', 'projection': ['m', 'k'], 'density': density_a},
                    {'name': 'B', 'projection': ['n', 'k'], 'density': density_b},
                    {'name': 'Z', 'projection': ['m', 'n'], 'output': True},
                ],
            }],
        }
    }


def run_lab4(sparse_yaml=None, density_a=None, density_b=None, mapping_yaml=None, **workload_kwargs):
    """Run a Lab 4 configuration and return the result.
    
    Args:
        sparse_yaml: Name of sparse config file in designs/ (e.g., 'sparse_gating.yaml')
        density_a: Override density of A (default 0.25)
        density_b: Override density of B (default 0.5)
        mapping_yaml: Path to mapping YAML (default: 'designs/mapping.yaml')
        **workload_kwargs: Extra keyword arguments forwarded to make_workload_yaml
                           (e.g., k=16 to set the K dimension)
    
    Returns:
        AccelForge Mappings result object
    """
    files = ['designs/arch.yaml']
    
    if density_a is not None or density_b is not None or workload_kwargs:
        da = density_a if density_a is not None else 0.25
        db = density_b if density_b is not None else 0.5
        wl = make_workload_yaml(da, db, **workload_kwargs)
        with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
            yaml.dump(wl, f)
            files.append(f.name)
    else:
        files.append('designs/workload.yaml')
    
    files.append(mapping_yaml or 'designs/mapping.yaml')
    if sparse_yaml:
        files.append(os.path.join('designs', sparse_yaml))
    
    spec = af.Spec.from_yaml(*files)
    return spec.evaluate_mapping()


def get_energy(result):
    """Get total energy in pJ."""
    return float(result.energy())


def get_cycles(result):
    """Get total latency in cycles."""
    return float(result.latency())


def get_component_energy(result, component):
    """Get energy for a specific component."""
    energy = result.energy(per_component=True)
    return float(energy.get(component, 0))

In [1]:
from accelforge import Spec

spec = Spec.from_yaml(
    "simple_arch.yaml",
    "spmspm.yaml",
    "spmspm_mapping.yaml",
)
result = spec.evaluate_mapping()

print("Energy (pJ):", float(result.energy()))
print("Latency (cycles):", float(result.latency()))
print("Per-component energy:", result.energy(per_component=True))

Energy (pJ): 424.0
Latency (cycles): 8.0
Per-component energy: {'GlobalBuffer': 320.0, 'MainMemory': 96.0, 'MAC': 8.0}


In [2]:
from pathlib import Path

template = Path("test1.yaml").read_text()

filled = (
    template
    .replace("{{NUM_HEADS}}", "32")
    .replace("{{MODEL_DIM}}", "4096")
    .replace("{{FFN_DIM}}", "16384")
)

# Also fill defaults if needed
filled = filled.replace("{{BATCH_SIZE}}", "1")
filled = filled.replace("{{CONTEXT_TOKENS}}", "128")
filled = filled.replace("{{VOCAB_SIZE}}", "50257")

Path("test1_7b.yaml").write_text(filled)

15577

In [3]:
%%writefile draft_decode_7b_kv.yaml
workload:
  rank_sizes:
    {% set BATCH_SIZE = BATCH_SIZE | default(1) %}
    {% set N_TOKENS = N_TOKENS | default(8192) %}
    {% set N_NEW_TOKENS = N_NEW_TOKENS | default(1) %}
    {% set VOCAB_SIZE = VOCAB_SIZE | default(50257) %}

    B: {{BATCH_SIZE}}
    M: {{N_NEW_TOKENS}}
    M_FULL: {{N_TOKENS}}

    # GPT-3 6.7B-style draft model dimensions
    H: 32
    E: 128
    F: 128
    D: 4096
    C: 16384
    J: 4096
    G: 4096
    S: {{VOCAB_SIZE}}

  bits_per_value: {All: 8}
  persistent_tensors: weight - Intermediates

  einsums:
  - {einsum: "I[b, m, d] = I_in[b, m, d]", is_copy_operation: True}

  - "V_new[b, m, h, e] = I[b, m, d] * WV[h, e, d]"
  - "K_new[b, m, h, e] = I[b, m, d] * WK[h, e, d]"
  - "Q_new[b, m, h, e] = I[b, m, d] * WQ[h, e, d]"

  - einsum: "QK[b, m, m_full, h] = Q_new[b, m, h, e] * K[b, m_full, h, e]"
    renames: {input: Q_new}
  - "QK_softmax[b, m, m_full, h] = QK[b, m, m_full, h]"

  - einsum: "AV[b, m, h, f] = QK_softmax[b, m, m_full, h] * V[b, m_full, h, E: f]"
    renames: {input: QK_softmax}
  - "Z[b, m, g] = AV[b, m, h, f] * WZ[h, f, g]"
  - "FFA[b, m, c] = Z[b, m, g] * WFFA[g, c]"
  - "FFB[b, m, j] = FFA[b, m, c] * WFFB[c, j]"

  # LM head used by the driver to account for vocab projection cost.
  # AccelForge does not numerically sample a token; the driver does that.
  - "LOGITS[b, m, s] = FFB[b, m, j] * W_vocab[s, j]"

renames:
  einsums:
  - name: default
    tensor_accesses:
    - name: input
      source: Inputs & Intermediates if len(All) == 3 else Inputs
      expected_count: 1
    - name: output
      source: Outputs
      expected_count: 1
    - name: weight
      source: ~(input | output)
      expected_count: 1 if len(All) == 3 else 0

Writing draft_decode_7b_kv.yaml


In [4]:
%%writefile target_verify_175b_kv.yaml
workload:
  rank_sizes:
    {% set BATCH_SIZE = BATCH_SIZE | default(1) %}
    {% set N_TOKENS = N_TOKENS | default(8192) %}
    {% set N_NEW_TOKENS = N_NEW_TOKENS | default(3) %}
    {% set VOCAB_SIZE = VOCAB_SIZE | default(50257) %}

    B: {{BATCH_SIZE}}
    M: {{N_NEW_TOKENS}}
    M_FULL: {{N_TOKENS}}

    # GPT-3 175B target model dimensions
    H: 96
    E: 128
    F: 128
    D: 12288
    C: 49152
    J: 12288
    G: 12288
    S: {{VOCAB_SIZE}}

  bits_per_value: {All: 8}
  persistent_tensors: weight - Intermediates

  einsums:
  - {einsum: "I[b, m, d] = I_in[b, m, d]", is_copy_operation: True}

  - "V_new[b, m, h, e] = I[b, m, d] * WV[h, e, d]"
  - "K_new[b, m, h, e] = I[b, m, d] * WK[h, e, d]"
  - "Q_new[b, m, h, e] = I[b, m, d] * WQ[h, e, d]"

  # Minimum-change verify model: the speculated block attends to the prior KV cache.
  # This follows the approximation in the upstream GPT-3 KV-cache examples.
  - einsum: "QK[b, m, m_full, h] = Q_new[b, m, h, e] * K[b, m_full, h, e]"
    renames: {input: Q_new}
  - "QK_softmax[b, m, m_full, h] = QK[b, m, m_full, h]"

  - einsum: "AV[b, m, h, f] = QK_softmax[b, m, m_full, h] * V[b, m_full, h, E: f]"
    renames: {input: QK_softmax}
  - "Z[b, m, g] = AV[b, m, h, f] * WZ[h, f, g]"
  - "FFA[b, m, c] = Z[b, m, g] * WFFA[g, c]"
  - "FFB[b, m, j] = FFA[b, m, c] * WFFB[c, j]"
  - "LOGITS[b, m, s] = FFB[b, m, j] * W_vocab[s, j]"

renames:
  einsums:
  - name: default
    tensor_accesses:
    - name: input
      source: Inputs & Intermediates if len(All) == 3 else Inputs
      expected_count: 1
    - name: output
      source: Outputs
      expected_count: 1
    - name: weight
      source: ~(input | output)
      expected_count: 1 if len(All) == 3 else 0

Writing target_verify_175b_kv.yaml


In [5]:
%%writefile target_correction_175b_kv.yaml
workload:
  rank_sizes:
    {% set BATCH_SIZE = BATCH_SIZE | default(1) %}
    {% set N_TOKENS = N_TOKENS | default(8192) %}
    {% set N_NEW_TOKENS = N_NEW_TOKENS | default(1) %}
    {% set VOCAB_SIZE = VOCAB_SIZE | default(50257) %}

    B: {{BATCH_SIZE}}
    M: {{N_NEW_TOKENS}}
    M_FULL: {{N_TOKENS}}

    # GPT-3 175B target model dimensions
    H: 96
    E: 128
    F: 128
    D: 12288
    C: 49152
    J: 12288
    G: 12288
    S: {{VOCAB_SIZE}}

  bits_per_value: {All: 8}
  persistent_tensors: weight - Intermediates

  einsums:
  - {einsum: "I[b, m, d] = I_in[b, m, d]", is_copy_operation: True}

  - "V_new[b, m, h, e] = I[b, m, d] * WV[h, e, d]"
  - "K_new[b, m, h, e] = I[b, m, d] * WK[h, e, d]"
  - "Q_new[b, m, h, e] = I[b, m, d] * WQ[h, e, d]"

  - einsum: "QK[b, m, m_full, h] = Q_new[b, m, h, e] * K[b, m_full, h, e]"
    renames: {input: Q_new}
  - "QK_softmax[b, m, m_full, h] = QK[b, m, m_full, h]"

  - einsum: "AV[b, m, h, f] = QK_softmax[b, m, m_full, h] * V[b, m_full, h, E: f]"
    renames: {input: QK_softmax}
  - "Z[b, m, g] = AV[b, m, h, f] * WZ[h, f, g]"
  - "FFA[b, m, c] = Z[b, m, g] * WFFA[g, c]"
  - "FFB[b, m, j] = FFA[b, m, c] * WFFB[c, j]"
  - "LOGITS[b, m, s] = FFB[b, m, j] * W_vocab[s, j]"

renames:
  einsums:
  - name: default
    tensor_accesses:
    - name: input
      source: Inputs & Intermediates if len(All) == 3 else Inputs
      expected_count: 1
    - name: output
      source: Outputs
      expected_count: 1
    - name: weight
      source: ~(input | output)
      expected_count: 1 if len(All) == 3 else 0

Writing target_correction_175b_kv.yaml
